# proof-engine quick start: comparing down-only designs

This notebook compares several contrasting down-only mortgage designs with a standard fixed-rate mortgage, in both lenses. **All numbers are arbitrary example inputs.** None of the designs is any company's product, and nothing here is an offer of credit.

In [ ]:
import numpy as np
from proof_engine import (
    BorrowerBehavior, DownOnlyLoan, FixedFloor, FloorBelowStart, HullWhiteMarket, PrepaymentModel,
    RefinancePolicy, StandardLoan, fair_rate_spread, par_spread_down_only, par_spread_standard,
    select_behavior, simulate,
)

## Simulated rates

A one-factor Hull–White model on a flat curve, on a daily grid. The main index is the 10-year model yield; a 2-year yield is added as a second named series that a design can follow instead.

In [ ]:
market = HullWhiteMarket(flat_rate=3.7, mean_reversion=0.11, volatility=1.3, index_tenor_years=10)
paths = market.simulate(months=360, days_per_month=21, paths=1_000, seed=21, antithetic=True,
                        series_tenors={"2y": 2})

## Five contrasting designs

Each is a `DownOnlyLoan`. `rate_spread` is ignored by the solvers below, which find the fair spread for each design.

In [ ]:
designs = {
    "One tier, no floor": DownOnlyLoan(
        rate_spread=1.6, triggers=(0.55,), check_every_days=1),
    "Two tiers, fixed floor": DownOnlyLoan(
        rate_spread=1.6, triggers=(0.35, 0.6), check_every_days=1, floor=FixedFloor(2.2)),
    "Three tiers, reset limit": DownOnlyLoan(
        rate_spread=1.6, triggers=(0.3, 0.45, 0.75), check_every_days=1, max_resets=4),
    "Prepayment penalty, floor below start": DownOnlyLoan(
        rate_spread=1.6, triggers=(0.55,), check_every_days=1, floor=FloorBelowStart(1.8),
        prepayment_penalty=(3.0, 2.0, 1.0)),
    "Averaged 2y index, notice lag, spacing": DownOnlyLoan(
        rate_spread=1.6, triggers=(0.4,), check_every_days=5, index_series="2y",
        index_average_days=20, notice_lag_days=10, min_days_between_resets=60),
}

## Borrower behavior

Behavior is an input for each borrower type. The type label only picks which of these inputs apply; the package has no built-in values for either type.

In [ ]:
behaviors = [
    BorrowerBehavior("owner_occupant",
                     prepayment=PrepaymentModel(0.08, 0.55, 0.9, 0.28, 0.9),
                     refinance=RefinancePolicy(threshold=0.75, delay_months=3, monthly_probability=0.3,
                                               requalify_probability=0.85, closing_costs=0.02, fixed_costs=1_000),
                     annual_turnover=0.08),
    BorrowerBehavior("investor",
                     prepayment=PrepaymentModel(0.12, 0.45, 0.8, 0.3, 0.9),
                     refinance=RefinancePolicy(threshold=0.6, delay_months=1, monthly_probability=0.55,
                                               requalify_probability=0.8, closing_costs=0.025, fixed_costs=1_500),
                     annual_turnover=0.12),
]
behavior = select_behavior(behaviors, "owner_occupant")

## Pricing lens: the cost of each design's feature

The par spread of each design minus the standard loan's par spread. The design following the 2-year series is priced on that series, so its spread is not directly comparable with the others.

In [ ]:
kw = dict(term_months=360, coupon_strip=0.55, oas=0.4, search=(-2.0, 6.0))
standard_spread = par_spread_standard(paths, prepayment=behavior.prepayment, **kw)
for name, loan in designs.items():
    spread = par_spread_down_only(paths, loan=loan, turnover_cpr=behavior.prepayment.turnover_cpr, **kw)
    note = "" if loan.index_series is None else "  (follows the 2y series)"
    print(f"{name:40s} {spread - standard_spread:+.3f} points{note}")

## Borrower-cost lens: the range of outcomes

For one borrower on each design, at the same spread as the standard loan, against the refinancing behavior above. Positive savings mean the down-only loan cost less on that path. Look at the whole range, including paths where it costs more.

In [ ]:
standard = StandardLoan(rate_spread=1.6, refinance=behavior.refinance)
borrower = dict(balance=300_000, term_years=30, horizon_years=10, discount_rate=3.1,
                annual_turnover=behavior.annual_turnover, seed=5)
for name, loan in designs.items():
    s = simulate(paths, down_only=loan, standard=standard, **borrower).summary()
    print(f"{name:40s} p5 {s['savings_p5']:>10,.0f}  median {s['savings_p50']:>10,.0f}  "
          f"p95 {s['savings_p95']:>10,.0f}  cheaper on {s['prob_down_only_cheaper']:.0%} of paths")

## Fair spread for one design

The down-only spread at which the borrower's expected cost equals the standard loan's.

In [ ]:
loan = designs["Three tiers, reset limit"]
fair = fair_rate_spread(paths, down_only=loan, standard=standard, search=(-2.0, 6.0), **borrower)
print(f"Fair spread: {fair:.3f} (standard loan: {standard.rate_spread:.3f})")